# 2 - Baselines de ML — KNN, MICE e MissForest

## Objetivo

Implementar e avaliar os **baselines de ML clássico** para imputação. Estes são os concorrentes diretos da GAIN: superar a média por estação é o mínimo; superar **estes** métodos é o que dá valor à Etapa 4.

> **Pergunta que responde:** quão bem se sai a imputação clássica de ML? Qual número a GAIN precisa bater para justificar sua complexidade?

## Posição na Etapa 3

Notebook **2 de 2** — fim da etapa de baselines. Consome os mesmos artefatos do notebook 1 + `tabelas/baselines_simples.csv`. Entrega os parquets de predições, `tabelas/baselines_ml.csv` e a **consolidação final** `tabelas/tabela_baselines.csv` (entregável da Etapa 3).

## Os 3 métodos

| Método | Intuição (o que assume) | Hiperparâmetro central |
|--------|-------------------------|------------------------|
| KNN-Imputer | Coletas próximas no espaço de features têm valores parecidos; a "vizinhança" usa todas as colunas disponíveis (inclusive estação e mês) | `n_neighbors=5, weights='distance'` |
| MICE (`IterativeImputer`) | Cada variável é uma função (linear-bayesiana) das demais; regressões iteradas até estabilizar | `max_iter=10`, estimador `BayesianRidge` |
| MissForest | Como MICE, mas com Random Forest — captura não-linearidades e interações sem supor forma funcional | `n_estimators=100, max_iter=10` |

## Protocolo

Idêntico ao notebook 1 (mesmo `baseline_utils.avaliar_baseline`, mesmas sementes `[42, 7, 2026]`, mesmo `miss_rate=0,20`): as máscaras `B` geradas por seed são **bit a bit as mesmas**, então as métricas são diretamente comparáveis entre os dois notebooks.

Diferença relevante: os métodos de ML enxergam, além das 11 variáveis-alvo, as **features de contexto** (temporais, one-hot de estação, one-hot de censura `_LD`) — que nunca têm NaN e atuam só como preditoras. Os imputadores são **ajustados no train** e aplicados ao test mascarado (protocolo indutivo — o mesmo da GAIN, que treina no train e imputa o test).

## Adaptações em relação ao plano (`Pipeline/03_Baselines/02_baselines_ml.md`)

- Mesmas adaptações de caminho e de nº de variáveis do notebook 1.
- **MissForest via `IterativeImputer(estimator=RandomForestRegressor)`**: o pacote `missingpy` não está instalado e está abandonado (incompatível com scikit-learn moderno — depende de APIs internas removidas). A emulação é fiel por construção: MissForest **é** imputação iterativa com Random Forest como regressor (Stekhoven & Bühlmann, 2012). Mantemos os hiperparâmetros do plano.

## Setup

Além dos imports do notebook 1: `enable_iterative_imputer` (o `IterativeImputer` ainda é experimental no scikit-learn e exige o import explícito), `KNNImputer` e `RandomForestRegressor`.

In [1]:
import json
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import IterativeImputer, KNNImputer

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

sys.path.append("../3 - Preprocessing")
from mask_utils import generate_artificial_mask

import baseline_utils as bu

IN_SPLIT_DIR    = Path("../../Data/GoldData/Splited")
IN_MASK_DIR     = Path("../../Data/GoldData/Masked")
IN_COLUMNS_JSON = Path("../../Data/ProcessedData/encoded_columns.json")
IN_SCALERS      = Path("../../Data/ProcessedData/scalers.pkl")
IN_TRANSFORM    = Path("../../Data/ProcessedData/transform_params.json")
IN_TAB_SIMPLES  = Path("./tabelas/baselines_simples.csv")

OUT_PRED_DIR = Path("../../Data/BaselineResults")
OUT_TAB_DIR  = Path("./tabelas")

SEEDS     = [42, 7, 2026]
MISS_RATE = 0.20

## Carregamento

Mesmo do notebook 1, mais a definição do **espaço de features** dos imputadores: `VARS` (alvo da imputação) + contexto (`temporais`, `estacao_onehot`, `ld_onehot`). Identificadores (`Data`, `Ano_int`) ficam de fora do vetor, como na GAIN.

In [2]:
with open(IN_COLUMNS_JSON, encoding="utf-8") as f:
    encoded_columns = json.load(f)
VARS = encoded_columns["numericas"]
CONTEXT_COLS = (encoded_columns["temporais"]
                + encoded_columns["estacao_onehot"]
                + encoded_columns["ld_onehot"])
FEATURES = VARS + CONTEXT_COLS

df_train = pd.read_parquet(IN_SPLIT_DIR / "train.parquet")
df_test  = pd.read_parquet(IN_SPLIT_DIR / "test.parquet")
mask_real_test = pd.read_parquet(IN_MASK_DIR / "mask_real_test.parquet")

# Pickle interno do projeto (gerado por 04_normalizacao.ipynb) — fonte confiável
scalers = joblib.load(IN_SCALERS)
with open(IN_TRANSFORM, encoding="utf-8") as f:
    transform_params = json.load(f)

# Sanidade: contexto não pode ter NaN (precondição — só as VARS são imputáveis)
assert df_train[CONTEXT_COLS].notna().all().all() and df_test[CONTEXT_COLS].notna().all().all(), \
    "Features de contexto com NaN — violaria o design das máscaras."

train_feat = df_train[FEATURES]
print(f"train: {df_train.shape} | test: {df_test.shape} | "
      f"features do imputador: {len(FEATURES)} ({len(VARS)} alvo + {len(CONTEXT_COLS)} contexto)")

train: (515, 36) | test: (90, 36) | features do imputador: 34 (11 alvo + 23 contexto)


### Adaptador comum

Os três métodos compartilham o mesmo fluxo: ajustar o imputador no `train` completo, montar a matriz de test com as VARS mascaradas + contexto intacto, transformar e devolver só as VARS. A fábrica recebe a `seed` para os métodos estocásticos (MICE e MissForest); o KNN é determinístico e a ignora.

In [3]:
def fazer_predict_sklearn(fabrica_imputador):
    def predict_fn(X_obs, seed):
        imputador = fabrica_imputador(seed)
        imputador.fit(train_feat)
        test_feat = pd.concat([X_obs, df_test[CONTEXT_COLS]], axis=1)
        out = imputador.transform(test_feat)
        return pd.DataFrame(out, columns=FEATURES, index=X_obs.index)[VARS]
    return predict_fn

## Método 1 — KNN-Imputer

Para cada célula faltante, encontra as `k=5` coletas mais próximas (distância euclidiana ignorando NaN, `nan_euclidean`) que têm a variável observada e tira a média **ponderada pela distância**. Assume que proximidade no espaço de features implica valores parecidos — e aqui a vizinhança é informada pela estação (one-hot) e pela sazonalidade (`Mes_sin/cos`), então o KNN pode ser lido como uma "média por estação-e-época" adaptativa.

`weights='distance'` suaviza a fronteira entre vizinhanças: vizinhos quase idênticos dominam a média. Com apenas ~515 coletas de train, `k=5` (~1% da base) é um compromisso razoável entre variância e viés local.

In [4]:
predict_knn = fazer_predict_sklearn(
    lambda seed: KNNImputer(n_neighbors=5, weights="distance"))

met_knn, pred = bu.avaliar_baseline(
    "knn", predict_knn, df_test, mask_real_test,
    VARS, scalers, transform_params, SEEDS, MISS_RATE, generate_artificial_mask)

pred.to_parquet(OUT_PRED_DIR / "predictions_knn.parquet", index=False)
met_knn.groupby("variavel", sort=False)[["RMSE", "MAE", "n_avaliado"]].mean().round(4)

,RMSE,MAE,n_avaliado
variavel,,,
DBO,8.7440,5.4502,14.3333
OD,3.5886,2.8638,12.3333
Nitrato,0.2003,0.1273,14.3333
Nitrogênio Amoniacal Total,2.7445,2.4044,13.3333
Fósforo Total,108.8210,34.0540,10.6667
Condutividade,9971.2317,6810.9523,13.6667
pH,0.4953,0.3992,17.6667
Turbidez,141.3695,44.9296,18.0000
Temperatura da Água,2.7928,2.1086,15.6667


## Método 2 — MICE (`IterativeImputer`)

*Multivariate Imputation by Chained Equations*: inicializa os faltantes com a média, depois itera regredindo cada variável sobre **todas as outras** (estimador padrão: `BayesianRidge`), atualizando as imputações a cada rodada até estabilizar (`max_iter=10`). Explora a estrutura de correlação global — exatamente o que a matriz de correlações da EDA (§3) mostrou existir entre DBO, Fósforo e Coliformes.

É o concorrente clássico mais citado contra a GAIN (é o baseline central do próprio paper do GAIN, Yoon et al. 2018). Limitação: as regressões são lineares — relações não-lineares ficam para o MissForest. Um `ConvergenceWarning` em 10 iterações é aceitável para baseline; silenciamos apenas esse aviso para não poluir a saída.

In [5]:
predict_mice = fazer_predict_sklearn(
    lambda seed: IterativeImputer(max_iter=10, random_state=seed))

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*did not converge.*")
    met_mice, pred = bu.avaliar_baseline(
        "mice", predict_mice, df_test, mask_real_test,
        VARS, scalers, transform_params, SEEDS, MISS_RATE, generate_artificial_mask)

pred.to_parquet(OUT_PRED_DIR / "predictions_mice.parquet", index=False)
met_mice.groupby("variavel", sort=False)[["RMSE", "MAE", "n_avaliado"]].mean().round(4)

C:\Users\jhter\OneDrive\Documents\QualiAgua\venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


C:\Users\jhter\OneDrive\Documents\QualiAgua\venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


C:\Users\jhter\OneDrive\Documents\QualiAgua\venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


,RMSE,MAE,n_avaliado
variavel,,,
DBO,13.0630,8.0129,14.3333
OD,2.6636,2.1218,12.3333
Nitrato,0.1596,0.0942,14.3333
Nitrogênio Amoniacal Total,16.4828,8.8250,13.3333
Fósforo Total,108.8878,34.0841,10.6667
Condutividade,6690.2505,4769.3531,13.6667
pH,0.6215,0.4433,17.6667
Turbidez,64.2902,28.5585,18.0000
Temperatura da Água,2.2537,1.7556,15.6667


## Método 3 — MissForest

Mesma iteração encadeada do MICE, trocando a regressão bayesiana por um **Random Forest** (`n_estimators=100`): captura não-linearidades e interações (ex.: "OD despenca quando temperatura sobe **e** a estação é a CM320") sem supor forma funcional. Historicamente é o método clássico mais forte em dados ambientais tabulares.

Como documentado no cabeçalho, usamos a emulação `IterativeImputer(estimator=RandomForestRegressor)` — a definição original do MissForest — em vez do pacote `missingpy`, abandonado e incompatível com o scikit-learn atual. Custo computacional: ~11 florestas × 10 iterações × 3 seeds; é o baseline mais lento (minutos, não segundos).

In [6]:
predict_missforest = fazer_predict_sklearn(
    lambda seed: IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=seed),
        max_iter=10, random_state=seed))

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*did not converge.*")
    met_missforest, pred = bu.avaliar_baseline(
        "missforest", predict_missforest, df_test, mask_real_test,
        VARS, scalers, transform_params, SEEDS, MISS_RATE, generate_artificial_mask)

pred.to_parquet(OUT_PRED_DIR / "predictions_missforest.parquet", index=False)
met_missforest.groupby("variavel", sort=False)[["RMSE", "MAE", "n_avaliado"]].mean().round(4)

C:\Users\jhter\OneDrive\Documents\QualiAgua\venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


C:\Users\jhter\OneDrive\Documents\QualiAgua\venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


C:\Users\jhter\OneDrive\Documents\QualiAgua\venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


,RMSE,MAE,n_avaliado
variavel,,,
DBO,9.5658,5.4889,14.3333
OD,3.0809,2.2004,12.3333
Nitrato,0.2162,0.1474,14.3333
Nitrogênio Amoniacal Total,2.7212,2.0621,13.3333
Fósforo Total,108.7866,34.0397,10.6667
Condutividade,7870.3200,6298.1597,13.6667
pH,0.4459,0.3584,17.6667
Turbidez,131.9779,42.2155,18.0000
Temperatura da Água,3.0417,2.2730,15.6667


## Consolidação da Etapa 3

Concatena os baselines simples (notebook 1) com os de ML e salva:

- `tabelas/baselines_ml.csv` — formato longo, só os 3 métodos deste notebook.
- `tabelas/tabela_baselines.csv` — **entregável final da Etapa 3**, agregado por (método, variável) no schema do plano (`metodo, variavel, RMSE_media, RMSE_std, MAE_media, MAE_std`) + colunas extras (`RMSE_norm_media`, `MAE_norm_media`, `n_avaliado_total`) que sustentam o ranking global e a leitura de ruído.

In [7]:
met_ml = pd.concat([met_knn, met_mice, met_missforest], ignore_index=True)
met_ml.to_csv(OUT_TAB_DIR / "baselines_ml.csv", index=False, encoding="utf-8")

met_simples = pd.read_csv(IN_TAB_SIMPLES, encoding="utf-8")
met_todos = pd.concat([met_simples, met_ml], ignore_index=True)

n_esperado = 7 * 10 * len(SEEDS)  # 7 métodos × 10 variáveis avaliáveis × 3 seeds
assert len(met_todos) == n_esperado, f"Esperava {n_esperado} linhas, obtive {len(met_todos)}."

tabela_final = bu.agregar_metricas(met_todos)
tabela_final.to_csv(OUT_TAB_DIR / "tabela_baselines.csv", index=False, encoding="utf-8")
print(f"tabela_baselines.csv salvo — {len(tabela_final)} linhas agregadas "
      f"({len(met_todos)} linhas brutas).")

# RMSE (escala original) por método × variável
pivot_rmse = tabela_final.pivot(index="variavel", columns="metodo", values="RMSE_media")
pivot_rmse["melhor_metodo"] = pivot_rmse.idxmin(axis=1)
pivot_rmse.round(3)

tabela_baselines.csv salvo — 70 linhas agregadas (210 linhas brutas).


metodo,ffill_estacao,knn,media_estacao,media_global,mediana_global,mice,missforest,melhor_metodo
variavel,,,,,,,,
Condutividade,14745.0670,9971.2320,10200.5090,15836.5420,17085.6090,6690.2510,7870.3200,mice
DBO,6.1710,8.7440,10.5860,10.7570,10.5760,13.0630,9.5660,ffill_estacao
Fósforo Total,108.8630,108.8210,108.8580,108.9210,108.8720,108.8880,108.7870,missforest
Nitrato,0.1430,0.2000,0.1640,0.1640,0.1670,0.1600,0.2160,ffill_estacao
Nitrogênio Amoniacal Total,4.0700,2.7450,3.0490,2.7740,2.8610,16.4830,2.7210,missforest
OD,3.6780,3.5890,4.0630,4.0230,3.9320,2.6640,3.0810,mice
Sólidos Suspensos Totais,39.5830,26.4650,31.9930,35.4280,33.5990,32.6780,41.1490,knn
Temperatura da Água,3.5510,2.7930,3.5090,3.4620,3.6780,2.2540,3.0420,mice
Turbidez,143.8180,141.3690,145.0200,153.4080,153.6530,64.2900,131.9780,mice


In [8]:
# Ranking global de todos os baselines da Etapa 3 (RMSE_norm médio, espaço [-1, 1])
ranking = (met_todos.groupby("metodo")[["RMSE_norm", "MAE_norm"]]
           .mean().sort_values("RMSE_norm"))
ranking.round(4)

,RMSE_norm,MAE_norm
metodo,,
knn,0.3127,0.2473
mice,0.3328,0.2577
missforest,0.3469,0.2672
media_estacao,0.3497,0.2869
media_global,0.3762,0.3135
mediana_global,0.3850,0.3168
ffill_estacao,0.3979,0.2830


In [9]:
# Melhor baseline por variável — os números que a GAIN precisa bater, um a um
melhor_por_var = (tabela_final
                  .loc[tabela_final.groupby("variavel")["RMSE_norm_media"].idxmin()]
                  [["variavel", "metodo", "RMSE_media", "RMSE_std", "MAE_media",
                    "RMSE_norm_media", "n_avaliado_total"]]
                  .set_index("variavel"))
melhor_por_var.round(4)

,metodo,RMSE_media,RMSE_std,MAE_media,RMSE_norm_media,n_avaliado_total
variavel,,,,,,
Condutividade,mice,6690.2505,1114.0784,4769.3531,0.2109,41
DBO,knn,8.7440,4.1096,5.4502,0.2088,43
Fósforo Total,missforest,108.7866,187.8540,34.0397,0.1621,32
Nitrato,media_global,0.1642,0.0496,0.0913,0.6194,43
Nitrogênio Amoniacal Total,missforest,2.7212,0.4012,2.0621,0.4635,40
OD,mice,2.6636,0.1658,2.1218,0.2558,37
Sólidos Suspensos Totais,knn,26.4653,8.4882,25.3426,0.2388,10
Temperatura da Água,mice,2.2537,1.0704,1.7556,0.2254,47
Turbidez,mice,64.2902,36.8227,28.5585,0.1728,54


## Síntese final da Etapa 3

### Ranking global (RMSE_norm médio, espaço [-1, 1])

| # | Método | RMSE_norm | MAE_norm |
|---|--------|-----------|----------|
| 1 | **knn** | **0,313** | **0,247** |
| 2 | mice | 0,333 | 0,258 |
| 3 | missforest | 0,347 | 0,267 |
| 4 | media_estacao | 0,350 | 0,287 |
| 5 | media_global | 0,376 | 0,314 |
| 6 | mediana_global | 0,385 | 0,317 |
| 7 | ffill_estacao | 0,398 | 0,283 |

### Leitura

- **O concorrente forte da GAIN é o KNN** (`RMSE_norm ≈ 0,313`), não o MICE que o plano antecipava. Com ~515 coletas de train e vizinhança informada por estação e sazonalidade, a "média local adaptativa" do KNN bate as regressões encadeadas. O MICE fica logo atrás (0,333) e **vence em mais variáveis individualmente** (Condutividade, OD, Temperatura, Turbidez) — os dois são referências legítimas.
- **MissForest não paga sua complexidade aqui** (0,347, apenas ~1% melhor que a média por estação no agregado): com 90 linhas de test e poucas centenas de train, o Random Forest não tem amostra para explorar não-linearidades sem sobreajustar. Ainda assim é o melhor em pH, Fósforo Total (no espaço normalizado) e Nitrogênio Amoniacal.
- **Nitrato resiste a tudo**: o melhor método é a **média global** (`RMSE_norm ≈ 0,62`, o pior "melhor" da tabela). O Box-Cox do Nitrato tem λ = −0,39 (inversa explosiva) e a variável tem cobertura intermitente — nenhuma correlação com as demais variáveis se traduz em imputação útil. Variável a observar com atenção no diagnóstico da GAIN.
- **Fósforo Total confirma o alerta duplo da Etapa 2**: no espaço normalizado o MissForest reduz bem o erro (0,162, o melhor RMSE_norm da tabela), mas na escala original o RMSE (~108,8 µg/L) é praticamente idêntico ao da média global — o erro físico é dominado pelos picos de eutrofização que nenhum método recupera. A comparação com a GAIN nesta variável deve olhar as **duas** escalas.
- Gap metodológico conhecido: todos os métodos ignoram a ordem temporal fina (só veem `Mes_sin/cos`, `ano_norm`, `dias_desde_inicio` como features estáticas). É onde a GAIN pode se diferenciar.

### Os números a bater na Etapa 4

- **Global**: `RMSE_norm < 0,313` (KNN). O `config_best.json` da GAIN só deve ser aceito se superar isso sob o **mesmo protocolo** (mesmas seeds `[42, 7, 2026]`, mesmo `miss_rate=0,20`, mesmas máscaras `B`).
- **Por variável**: tabela `melhor_por_var` acima (persistida em `tabelas/tabela_baselines.csv`).

### Critério de aceite

- ✅ 3 arquivos de predições de ML em `Data/BaselineResults/` (7 no total da etapa).
- ✅ `tabelas/tabela_baselines.csv` consolida os 7 métodos × 10 variáveis avaliáveis.
- ✅ Concorrente forte declarado: **KNN** (com MICE como segunda referência).

## Próxima etapa

`04_GAIN/01_treino.ipynb` — precedido pela implementação do módulo `gain.py`. Reutilizar `baseline_utils.avaliar_baseline` na Etapa 5 para garantir comparabilidade.